# Lean-19 : La Conjecture de Sendov (T. Tao, aout 2026) — Digestion pedagogique

**Serie** : SymbolicAI / Lean — Digestions de resultats profonds
**Auteur source** : Terence Tao, 2026-08-12
**Lac source** : https://github.com/teorth/sendov
**Blog de reference** : https://terrytao.wordpress.com/2026/08/12/a-digestion-of-the-proof-of-sendovs-conjecture/

## Navigation

| Notebook precedent | Notebook suivant |
|---|---|
| [Lean-18 - Recherche A* Optimalite](../Lean-18-Search-AStar-Optimality.ipynb) | Lean-20 (a venir) |

---

## Presentation

Ce notebook presente la **conjecture de Sendov** et sa preuve par Terence Tao, telle que formalisee en Lean 4 dans le lac [teorth/sendov](https://github.com/teorth/sendov) (aout 2026). C'est la **digestion** d'un nouveau grand theoreme en analyse complexe, demontree par Tao en 2 jours avec Claude Opus 5 et publiee sous Apache 2.0.

**La conjecture de Sendov** dit : *Si p in C[X] a degre n >= 2 avec tous ses zeros dans le disque unite ferme, alors pour chaque zero a de p, il existe un point critique zeta de p a distance |zeta - a| <= 1.* La conjecture plus forte de **Phelps-Rodriguez** exige |zeta - a| < 1 sauf dans le cas extreme ou |a| = 1 et p est un multiple scalaire de z^n - a^n.

**Pourquoi ce notebook dans notre serie Lean ?**

- Notre serie Lean (notebooks 1 a 18) va du langage lui-meme (Lean-1..6 : types dependants, propositions, quantificateurs, tactiques, Mathlib) a des theoremes profonds et recents : Huang Sensitivity (Lean-12), Kochen-Specker (Lean-13), Grothendieck Tribute (Lean-15), Conway Knots (Lean-17), A* Optimalite (Lean-18).
- Sendov s'inscrit dans la meme veine : un theoreme profond recemment demontre (ici par Tao en digestion d'une preuve de Mazur), avec une formalisation Lean propre et bien ficelee, qui merite le meme traitement pedagogique qu'un Huang ou un Conway.
- **Substance nouvelle** : on n'avait pas encore aborde l'analyse complexe dans notre serie. Sendov ouvre cette porte avec un objet mathematique riche - polynomes sur C, integrales, inegalites AM-GM, fonctions hyperboliques - qui complete notre palette apres la theorie des categories (Lean-15b).
- **Methode nouvelle** : pour la premiere fois dans notre serie, nous presentons une formalisation Lean qui **n'est pas de notre cru**. Le lac source est externe (teorth/sendov). C'est un **nod** modeste a Terry Tao, qui pousse depuis 2 ans pour l'adoption de la preuve agentique - un travail dont nous partageons l'esprit.

**Note methodologique** : conformement a la convention de notre serie (cf. Lean-12 Sensitivity, Lean-17 Knots), ce notebook utilise un **kernel Python 3**, pas Lean 4. Les enonces Lean sont presentes sous forme pedagogique (pseudo-Lean), et les preuves sont illustrees en Python. Le **vrai code Lean** est disponible dans le lac source : `git clone https://github.com/teorth/sendov && cd sendov && lake build Sendov`.

## 1. Enonces formels

### 1.1 La conjecture de Sendov

Soit p in C[X] un polynome de degre n >= 2 dont **tous les zeros sont dans le disque unite ferme** D_bar = {z in C : |z| <= 1}. Alors pour **chaque zero a** de p, il existe un **point critique** zeta de p tel que :

    |zeta - a| <= 1

Rappel : un point critique de p est un zero de p' (la derivee formelle du polynome).

### 1.2 La conjecture de Phelps-Rodriguez (plus forte)

Sous les memes hypotheses, sauf dans le cas extreme ou |a| = 1 **et** p = c * (z^n - a^n) pour un certain scalaire c != 0, on a l'inegalite stricte :

    |zeta - a| < 1

Le cas extreme correspond exactement aux **polynomes extremaux** de Rubinstein, ou la borne <= 1 est atteinte (et la distance vaut exactement 1).

### 1.3 Pourquoi c'est non-trivial

A premiere vue, on pourrait penser qu'un polynome dont tous les zeros sont dans D_bar a ses points critiques egalement dans D_bar. C'est **faux en general** - par exemple, p(z) = z^n - 1 a tous ses zeros sur le cercle unite et tous ses points critiques en 0. La conjecture de Sendov dit qu'on a un *controle plus subtil* : chaque zero a a *au moins un* point critique zeta dans la boule B_bar(a, 1) de centre a et de rayon 1.

C'est cette geometrie locale qui rend la conjecture profonde et qui justifie une preuve cas par cas.

In [1]:
# Code 1.1 - Enonce Sendov (format pedagogique pseudo-Lean)
#
# On reproduit l'enonce du lac source Sendov/Conjecture.lean :104-110
# avec les memes types mais une syntaxe simplifiee pour la clarte.

def sendov_statement(n, p, a, all_roots_in_unit_disk):
    """
    Conjecture de Sendov (Tao 2026, formalisation Lean 4).

    Parametres :
        n (int) : degre du polynome, >= 2
        p (Polynomial) : polynome a coefficients complexes, degre n
        a (complex) : un zero de p (p(a) = 0)
        all_roots_in_unit_disk (bool) : tout zero w de p satisfait |w| <= 1

    Sortie : un point critique zeta de p avec |zeta - a| <= 1.
    """
    # Pseudo-Lean :
    # theorem sendov {n : Nat} (hn : 2 <= n) {p : C[X]} (hdeg : p.natDegree = n)
    #   (hroots : forall w in p.roots, |w| <= 1) {a : C} (hpa : p.eval a = 0) :
    #   exists zeta : C, (derivative p).eval zeta = 0 /\ |zeta - a| <= 1
    #
    # Implementation : voir Sendov/Conjecture.lean (lignes 104-110)
    # Preuve : voir c.8267+1 (cycles suivants)
    pass  # stub pedagogique (regle C.1 - pas de raise)

print("Sendov statement - voir Sendov/Conjecture.lean du lac teorth/sendov")
print("Formalisation complete : 14920 LOC, 76 fichiers, sorry-free")

Sendov statement - voir Sendov/Conjecture.lean du lac teorth/sendov
Formalisation complete : 14920 LOC, 76 fichiers, sorry-free


## 2. Architecture de la preuve

La preuve de Sendov + Phelps-Rodriguez procede par **cas sur la position du zero a** dans D_bar. Trois cas, plus un recollement.

### 2.1 Les quatre cas

| Cas | Position de a | Resultat invoque | Idee-cle |
|---|---|---|---|
| **Centre** | a = 0 | `Sendov.sendov_center` | Argument **produit de normes** : si tous les zeros w_i ont |w_i| <= 1, alors le produit des distances critiques-zeros satisfait une borne. |
| **Interieur** | 0 < |a| < 1 | `Sendov.sendov_interior_real` | On rotate a vers un reel r in (0,1) (changement de variable z -> z/omega). Puis on oppose deux **canaux** : canal polaire (qui utilise une integrale angulaire) et canal d'origine (qui utilise le fait que 0 est dans D_bar). Ils forcent une borne J_4(a) < 1 qui contredit 1 <= J_4(a). |
| **Frontiere** | |a| = 1 | `Sendov.rubinstein_one` | Cas de **Rubinstein** : la borne |zeta - a| <= 1 est atteinte exactement pour les polynomes extremaux c(z^n - 1). Phelps-Rodriguez dit que ces cas sont les **seuls** ou l'inegalite stricte < 1 echoue. |
| **Recollement** | Position quelconque | `Sendov.Conjecture.phelps_rodriguez` | Elimine la normalisation a in [0,1) par **rotation complexe** a = omega*r, transport des points critiques par omega^-1. |

### 2.2 Subtilite technique : pourquoi 3 canaux pour 1 conjecture ?

Le cas interieur est de loin le plus technique. Il mobilise une inegalite de **Maclaurin** (sur les sommes symetriques), un **lemme de defaut** sur les produits dans le disque unite, et une borne sur log(sinh h / h) <= sqrt(h^2 + 9) - 3 qui donne le controle precis necessaire.

Pour les degres n = 2, 3, 4, 5, Tao utilise des **certificats de Bernstein** rationnels (pas de flottants !) qui verifient numeriquement la borne. Pour n >= 5, c'est l'argument analytique general. Pour n >= 101, l'argument est different (canal large degre).

### 2.3 Le fichier `Conjecture.lean` (envoi final)

Le fichier `Sendov/Conjecture.lean` du lac source est remarquable : il ne fait **que** recoller. Sa preuve se lit comme une partition par cas, et chaque cas est deja prouve dans un autre module. C'est un bel exemple d'architecture en theorie des types : **decomposition modulaire** + recollement par `rcases` / `Or.inl` / `Or.inr`.

## 3. Bibliographie Mathlib minimale

Pour comprendre la preuve, le **minimum Mathlib** a connaitre est :

### 3.1 Tactic automation

- `Mathlib.Tactic.Linarith` - resolution d'inegalites lineaires sur les reels (massivement utilise pour les bornes de Maclaurin et les produits de normes).
- `Mathlib.Tactic.Positivity` - prouve automatiquement qu'une expression est >= 0 ou > 0.
- `Mathlib.Tactic.Ring` - egalites polynomiales closes.
- `Mathlib.Tactic.FieldSimp` - manipulation d'expressions fractionnaires.

### 3.2 Analyse complexe

- `Mathlib.Analysis.Complex.Basic` - operations de base sur C, conjugue, norme.
- `Mathlib.Analysis.Complex.Polynomial.Basic` - zeros et racines dans C.
- `Mathlib.Analysis.Complex.ExponentialBounds` - bornes sur |e^z| utiles pour les integrales.

### 3.3 Fonctions speciales

- `Mathlib.Analysis.SpecialFunctions.Log.Deriv` - derivee de log, utilisee dans `Sendov.Analytic.Polar`.
- `Mathlib.Analysis.SpecialFunctions.Trigonometric.DerivHyp` - derivee de sinh, au coeur du lemme log(sinh h / h) <= sqrt(h^2+9) - 3.
- `Mathlib.Analysis.SpecialFunctions.Pow.Real` - pour r^alpha dans les integrales.
- `Mathlib.Analysis.SpecialFunctions.Sqrt` - la borne sqrt(h^2+9) est partout.
- `Mathlib.Analysis.SpecialFunctions.Integrals.Basic` - integrales integral_0^1 f(t) dt qui sont au coeur du canal polaire.

### 3.4 Theorie de la mesure

- `Mathlib.MeasureTheory.Integral.IntervalIntegral.Basic` - integral_0^1 comme un cas particulier.
- `Mathlib.MeasureTheory.Integral.IntervalIntegral.FundThmCalculus` - le **theoreme fondamental** de l'analyse pour les integrales, indispensable pour passer d'une borne sur f a une borne sur integral f.

### 3.5 Algebre et polynomes

- `Mathlib.Algebra.Polynomial.Derivative` - la derivee formelle p' du polynome.
- `Mathlib.Algebra.Order.Ring.Pow` - pour les puissances dans les sommes symetriques (Maclaurin).
- `Mathlib.RingTheory.MvPolynomial.Symmetric.Defs` - les **polynomes symetriques** sous-jacents a l'inegalite de Maclaurin.

**Note de cadrage** : ces 20 imports Mathlib (cartographie verbatim c.8266-L1) sont remarquablement **compacts** par rapport a un depot de theorie des categories (qui consomme des centaines d'imports via `Mathlib.CategoryTheory.*`). C'est ce qui rend la preuve de Sendov digestible en 14.9k LOC, contre les 30-50k LOC habituelles pour une formalisation equivalente en categorie.

## 4. References croisees dans notre serie

Sendov a des **ponts naturels** avec plusieurs notebooks de notre serie :

### 4.1 Avec Lean-12 Sensitivity (Huang 2019)

Les deux sont des **digestions de theoremes profonds recents** : Huang (sensitivity conjecture de 1992 resolue en 2019) et Sendov (conjecture de 1959 resolue par Mazur, digestion Tao 2026). Memes principes methodologiques :

- Enonce en pseudo-Lean, execution en Python pour la visualisation.
- Architecture de la preuve decomposee en cas.
- Focus sur la **comprehension de la structure** plutot que sur les details tactiques.

### 4.2 Avec Lean-13 Kochen-Specker

Kochen-Specker est un theoreme de **logique** (mecanique quantique). Sendov est un theoreme d'**analyse**. Les deux partagent :

- Une intuition geometrique forte (spheres, disques, distances).
- Une preuve qui **divise par cas** sur la position d'un objet geometrique.

### 4.3 Avec Lean-15b Grothendieck Tribute

Lean-15b presente la **theorie des categories**. Sendov n'utilise pas de theorie des categories, mais le **recollement par rotation** dans `Sendov/Conjecture.lean` est un cas particulier de l'idee categorielle de **transport de structure** par un isomorphisme. Un etudiant qui voit Sendov apres Lean-15b peut lire le recollement et y voir un embryon d'adjonction : la rotation omega -> omega^-1 est une **involution naturelle** sur C*.

### 4.4 Avec Lean-18 Search A* Optimalite

Lean-18 presente l'**optimalite de A***. Sendov presente l'**optimalite d'un argument de Maclaurin** dans le cas interieur. Les deux reposent sur des bornes *precises* (pas seulement asymptotiques), ou il faut comprendre pourquoi la constante est exactement 1.

In [2]:
# Code 4.1 - Reference : envoyer vers Lean-12 / Lean-15b / Lean-17 / Lean-18
#
# Ce stub est conserve pour le grain c.8267+1, ou nous ajouterons
# la section 5 (cas du centre) avec une mini-implementation Python
# illustrant sendov_center sur des exemples explicites.

def sendov_center_pedagogic_example():
    """
    Exemple pedagogique (grain futur) : pour p(z) = z^2, le seul zero est 0,
    et le seul point critique est aussi 0 (p'(z) = 2z, donc p'(0) = 0).
    Distance |0 - 0| = 0 <= 1 : OK.

    Pour p(z) = z^3, zeros = {0}, points critiques = {0} (p'(z) = 3z^2, racine 0).
    Distance = 0 <= 1 : OK.

    Pour p(z) = z^n - 1, zeros = les racines n-iemes de 1, points critiques
    sont tous en 0 (p'(z) = n z^(n-1)). Distance d'une racine w a 0 vaut |w| = 1
    exactement : cas extreme de Rubinstein, ou l'inegalite stricte echoue.
    """
    # Implementation : voir c.8267+1
    pass  # stub pedagogique (regle C.1)

print("Lean-19 Sendov : skeleton termine - grain 1/2")
print("Voir issue #10759 pour le plan complet (grain 2 = cas du centre, interieur, boundary)")

Lean-19 Sendov : skeleton termine - grain 1/2
Voir issue #10759 pour le plan complet (grain 2 = cas du centre, interieur, boundary)
